# Uninformed Search

In [1]:
# this is something we will run each time at the beginning of our notebook
# in order to access the modules present in scripts
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "scripts").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

# and then we import the module as scripts.module_name
from scripts.environment import *
from scripts.agent import Agent
from scripts.runner import run_episode

In this notebook we will work with the uninformed search.\
We will do a simple warmup exercise where we try to understand the importance of using determined data structure depending on the particular case, and then we will try to implement some planning agent on our Vacuum World.

To begin, let's briefly recap the theory behind classical search.

## The planning agent and the search problem

Unlike a reflex agent that chooses actions solely based on the current percept, a planning agent uses an internal model to explore potential action sequences.

We assume, from now on, that the environment is single-agent, fully observable, static (no external changes while planning), has discrete states with deterministic actions, and follows a sequential task structure where an action impacts later possibilities.

A search problem is defined by the following components:

* State Space ($S$): The set of all possible states.
* Available Actions ($A$): The actions available for a given state $s$, denoted as $A: S \rightarrow 2^{\mathcal{A}}$. 
* Transition Model ($T$): Returns the successor state given a state and action, $T: S \times \mathcal{A} \rightarrow S$.
* Step Cost ($C$): The numerical cost of taking an action, $c: S \times \mathcal{A} \times S \rightarrow \mathbb{R}_{\ge 0}$.
* Initial State ($s_0$): The starting state of the agent. 
* Goal Test: A function that determines whether a given state is a goal state ($S \rightarrow \{0, 1\}$).

We therefore define:

* **Plan**: it is an action sequence $\pi = (a_0, \dots, a_{k-1})$ that transitions from the initial state to a goal state.
* **Path cost**: it is the sum of step costs along the path: $g(\pi) = \sum_{i=0}^{k-1} c(s_i, a_i, s_{i+1})$. An optimal plan minimizes this cost.

The goal is usually not just to find a **valid plan**, but the one with ne **minimum path cost**.

Planning is based on a model of the environment which retains enough informations. In particular, a state must retain enough information to determine legal actions, successor states, step costs, and whether the goal is met. Past costs belong to the search node, not the state itself, unless they affect future possibilities.

The number of possible states available depends on the type of problems and how different states interacts with each others.

The model itself can be represented usually by two type of representation:

* **State-Space Graph**: Contains one vertex per state and edges representing legal actions. Loops and cycles can create infinite paths.
* **Search Tree**: Unfolds the state-space graph into a tree where each node represents an entire path from the start state. Different nodes can point to the same underlying state.

The **frontier** is an important auxiliary object used while doing the exploration of the states. It job is to store candidate nodes awaiting selection. Its underlying data structure defines the search strategy (what we call "the algorithm"):

* Stack (LIFO): Used for Depth-First Search (DFS).
* Queue (FIFO): Used for Breadth-First Search (BFS).
* Min-Priority Queue: Used for Uniform-Cost Search (UCS).

## Algorithms

Just for you to remember, here we report the algorithms seen during lecture to do uninformed search:

* **Deep First Search** (DFS)
* **Breadth First Search** (BFS)
* **Iterative Deepening Search** (IDS) 
* **Uniform Cost Search** (UCS)

We also evaluated them on completeness, optimality, time complexity, and space complexity.

| Criterion / Method | DFS | BFS | IDS | UCS |
| :--- | :--- | :--- | :--- | :--- |
| **Complete?** | No (in general) | Yes (finite branching) | Yes (finite branching) | Yes (under stated conditions) |
| **Cost-Optimal?** | No | Yes (equal positive costs) | Yes (equal positive costs) | Yes (non-negative costs) |
| **Time Complexity** | O(b^m) | O(b^(d+1)) | O(b^d) | O(b^(1 + floor(C*/epsilon))) |
| **Space Complexity** | O(bm) | O(b^(d+1)) | O(bd) | O(b^(1 + floor(C*/epsilon))) |

## Warmup Exercise

We want to understand how choosing a Stack (LIFO), a Queue (FIFO), or a Min-Priority Queue changes the order in which a search algorithm expands nodes.

Complete the following snippet of code inserting the elements saved into `items` inside `stack`, `queue` and `pq`. Use:

* a standard **Python list** as a Stack (LIFO)
* **collections.deque** as a Queue (FIFO) 
* **heapq** as a Min-Priority Queue (UCS)

In [2]:
from collections import deque
import heapq

items = [(4, "A"), (1, "B"), (3, "C")]

# Stack (LIFO) -> DFS
stack = []
for item in items:
  stack.append(item)
print("DFS Next Out:", stack.pop())  # (3, 'C') - Last in, first out

# Queue (FIFO) -> BFS
queue = deque()
for item in items:
  queue.append(item)
print("BFS Next Out:", queue.popleft())  # (4, 'A') - First in, first out

# Min-Priority Queue -> UCS
pq = []
for item in items:
  heapq.heappush(pq, item)  # heapq automatically sorts by the first element (cost)
print("UCS Next Out:", heapq.heappop(pq))  # (1, 'B') - Smallest cost first

DFS Next Out: (3, 'C')
BFS Next Out: (4, 'A')
UCS Next Out: (1, 'B')


An interesting aspect we want to consider is also the fact that the complexity of each operation is not always the same.

* Stack (LIFO / DFS): Insertions and removals operate in $O(1)$ time.
* Queue (FIFO / BFS): Insertions and removals operate in $O(1)$ time (using head/tail pointers).
* Min-Priority Queue (UCS / Heap): Insertions and removals operate in $O(\log F)$ time, where $F$ is the number of entries in the frontier, due to binary heap reordering.

Let's try to measure this!

In [3]:
import random
import time

# Number of elements to push and pop
n_operations = 100000
# Random Data
data = [(random.randint(1, 1000), f"node_{i}") for i in range(n_operations)]

# Benchmark Stack (LIFO / DFS) ---
start_time = time.perf_counter()
stack = []
for item in data:
  stack.append(item)  # O(1)
while stack:
  stack.pop()  # O(1)
stack_time = time.perf_counter() - start_time

# Benchmark Queue (FIFO / BFS) ---
start_time = time.perf_counter()
queue = deque()
for item in data:
  queue.append(item)  # O(1)
while queue:
  queue.popleft()  # O(1)
queue_time = time.perf_counter() - start_time

# Benchmark Min-Priority Queue (UCS / Heap) ---
start_time = time.perf_counter()
pq = []
for item in data:
  heapq.heappush(pq, item)  # O(log F)
while pq:
  heapq.heappop(pq)  # O(log F)
pq_time = time.perf_counter() - start_time

print(f"Benchmark Results on {n_operations:,} operations")
print(f"Stack (LIFO - List):      {stack_time:.4f} seconds")
print(f"Queue (FIFO - Deque):     {queue_time:.4f} seconds")
print(f"Priority Queue (Heap):    {pq_time:.4f} seconds")

Benchmark Results on 100,000 operations
Stack (LIFO - List):      0.0055 seconds
Queue (FIFO - Deque):     0.0045 seconds
Priority Queue (Heap):    0.0586 seconds


## Planning Agent

Time to code our planning agent!\
We will use a tree as data structure, in order to do it we will use a simple auxiliary object we will call `SearchNode`.

In [4]:
from collections import deque
import heapq

class SearchNode:
  """Represents a node in the search tree (a specific path)."""

  def __init__(self, state, parent=None, action=None, path_cost=0.0):
    self.state = state    # Strongly suggest to be in the form ((x, y), frozenset(dirts))
    self.parent = parent  # Previous SearchNode
    self.action = action  # Action taken to get here
    self.path_cost = path_cost  # Cumulative cost (g)

  # Required for priority queue comparisons in UCS
  # This is the dunder method of "less then"
  def __lt__(self, other):
    return self.path_cost < other.path_cost

You must try to implement everything we have described until now.\
You will create a `search` method which, given the initial state and the environment, builds up the search tree, returning a valid path to the goal state.\ Notice, you will need to simulate the environment inside the search method. Use the `_get_legal_actions`, `_transition` and `_reconstruct_path` helpers.\
Notice, this implementation of the agent don't use directly the `env` as helper. This may results in a less reusable agent, but it is a good way to separate the logic of the planning from the environment itself! 

In [5]:
class PlanningAgent(Agent):

  def __init__(self, env, search_strategy='bfs'):
    super().__init__(env)
    self.env = env
    self.search_strategy = search_strategy
    self.plan = []

  def act(self, observation):
    # If we already computed a plan, pop and return the next action
    if self.plan:
      return self.plan.pop(0)

    # Otherwise, run search to find the plan
    self.plan = self.search()
    if self.plan:
      return self.plan.pop(0)
    return 4  # Fallback to IDLE if no plan found

  def search(self):
    """Core search routine to build the search tree and find a path to the goal."""
    # Get initial state from the environment
    initial_pos = self.env.vacuum
    initial_dirts = frozenset(self.env.dirts)
    initial_state = (initial_pos, initial_dirts)

    # Create the root node
    root_node = SearchNode(state=initial_state, path_cost=0.0)

    # Initialise the frontier based on strategy
    # - BFS: deque (FIFO)
    # - DFS: list (LIFO stack)
    # - UCS: heapq (Min-Priority Queue)
    if self.search_strategy == 'bfs':
      frontier = deque([root_node])
    elif self.search_strategy == 'dfs':
      frontier = [root_node]
    elif self.search_strategy == 'ucs':
      frontier = []
      heapq.heappush(frontier, root_node)

    explored = set()

    while frontier:
      # POP NODE FROM FRONTIER
      if self.search_strategy == 'bfs':
        node = frontier.popleft()
      elif self.search_strategy == 'dfs':
        node = frontier.pop()
      elif self.search_strategy == 'ucs':
        node = heapq.heappop(frontier)  # heapq pop

      # GOAL TEST
      vacuum_pos, current_dirts = node.state
      if len(current_dirts) == 0:
        # Goal reached! Reconstruct action sequence (plan)
        return self._reconstruct_path(node)

      if node.state in explored:
        continue
      explored.add(node.state)

      # EXPAND NODE & GENERATE SUCCESSORS
      # Actions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT, 4=IDLE, 5=SUCK
      for action in self._get_legal_actions(vacuum_pos, current_dirts):
        next_state, step_cost = self._transition(vacuum_pos, current_dirts, action)
        
        child_node = SearchNode(
            state=next_state,
            parent=node,
            action=action,
            path_cost=node.path_cost + step_cost
        )

        if child_node.state not in explored:
          if self.search_strategy == 'bfs':
            frontier.append(child_node)
          elif self.search_strategy == 'dfs':
            frontier.append(child_node)
          elif self.search_strategy == 'ucs':
            heapq.heappush(frontier, child_node)

    return []

  def _get_legal_actions(self, vacuum_pos, current_dirts):
    """Helper mirroring env.get_possible_actions() using local state."""
    x, y = vacuum_pos
    possible = {
        0: (x, y - 1),
        1: (x, y + 1),
        2: (x - 1, y),
        3: (x + 1, y),
        4: (x, y),
        5: (x, y),
    }
    legal = []
    for action, (nx, ny) in possible.items():
      if action in (4, 5):
        legal.append(action)
        continue
      # Check grid bounds and walls (accessed via env in this case)
      if (0 <= nx < self.env.setting.grid_width and
              0 <= ny < self.env.setting.grid_height and
              (nx, ny) not in self.env.walls):
        legal.append(action)
    return legal

  def _transition(self, vacuum_pos, current_dirts, action):
    """Simulates the environment transition model T(s, a) -> s'."""
    x, y = vacuum_pos
    directions = {0: (0, -1), 1: (0, 1), 2: (-1, 0), 3: (1, 0)}
    
    new_pos = vacuum_pos
    new_dirts = set(current_dirts)
    step_cost = 1.0  # default cost

    if action in directions:
      dx, dy = directions[action]
      new_pos = (x + dx, y + dy)
      if new_pos in env.water_tiles:
        step_cost = 5 # moving on water costs more
    elif action == 5:  # SUCK
      if vacuum_pos in new_dirts:
        new_dirts.remove(vacuum_pos)
        step_cost = -10.0  # Reward/cost adjustment

    return (new_pos, frozenset(new_dirts)), step_cost

  def _reconstruct_path(self, node):
    """Trace parent pointers from goal back to root to form action sequence."""
    path = []
    while node.parent is not None:
      path.append(node.action)
      node = node.parent
    path.reverse()
    return path

Let's see it in action!\
Notice, you will probably not even be able to see it at difficulty 0 because is too fast. Try it directly on difficulty 1.

In [6]:
env = VacuumWorld(difficulty=1)
agent = PlanningAgent(env, "ucs")
run_episode(env, agent)

197.5

What about difficulty 2?\
Difficulty 2 insert some special tiles where moving costs more. Does your agent find a solution? Do you think it is an optimal solution?

In [7]:
env = VacuumWorld(difficulty=2)
agent = PlanningAgent(env, "ucs")
run_episode(env, agent)

193.39999999999998